In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

df = pd.read_csv("/content/all_universities_data_s_0_0_200_advanced.csv")
df = df[df['Стоимость'] != 0]
df['Стоимость'] = (df['Стоимость'] / 1000).round() * 1000
# Преобразование даты в числовые признаки
df['Дата архивации'] = pd.to_datetime(df['Дата архивации'])
df['Год'] = df['Дата архивации'].dt.year
df['Месяц'] = df['Дата архивации'].dt.month
df['День'] = df['Дата архивации'].dt.day


# Выбор нужных признаков
features = ['Год', 'Месяц', 'День', 'Проходной балл', 'Бюджетные места', 'Программа', 'Вуз']
X = df[features]
y = df['Стоимость'].values.reshape(-1, 1)  # Преобразуем в 2D-массив

# Масштабируем целевую переменную
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

# Разделяем признаки на числовые и категориальные
numerical_features = ['Год', 'Месяц', 'День', 'Проходной балл', 'Бюджетные места']
categorical_features = ['Программа', 'Вуз']

# Препроцессинг
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)
X_processed_dense = X_processed.toarray() if hasattr(X_processed, 'toarray') else X_processed

In [ ]:
X_reshaped = np.expand_dims(X_processed_dense, axis=1)  # (n_samples, 1, n_features)
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y_scaled, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    LSTM(64, input_shape=(1, X_processed_dense.shape[1]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - loss: 0.3208 - val_loss: 0.0633
Epoch 2/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0608 - val_loss: 0.0335
Epoch 3/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0338 - val_loss: 0.0209
Epoch 4/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 0.0232 - val_loss: 0.0153
Epoch 5/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0196 - val_loss: 0.0127
Epoch 6/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 0.0176 - val_loss: 0.0106
Epoch 7/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 0.0154 - val_loss: 0.0095
Epoch 8/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 0.0136 - val_loss: 0.0096
Epoch 9/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0126 - val_loss: 0.0085
Epoch 10/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0123 - val_loss: 0.0076
Epoch 11/50
2466/2466 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0114 - val_loss: 0.0076
Epoch 12/50
2466/24

In [ ]:
from sklearn.pipeline import Pipeline
import joblib
import pickle
from tensorflow.keras.models import load_model
# Сохраняем ВЕСЬ препроцессор (важно!)
joblib.dump(preprocessor, 'full_preprocessor_v1.joblib')

# Сохраняем модель
model.save('lstm_model_v1.keras')  # Новый рекомендуемый формат

# Сохраняем scaler_y
joblib.dump(scaler_y, 'scaler_y_v1.joblib')

# Сохраняем список признаков
with open('features_list_v1.pkl', 'wb') as f:
    pickle.dump(features, f)

In [ ]:
def load_model_components():
    try:
        # Загружаем в обратном порядке
        with open('features_list.pkl', 'rb') as f:
            features = pickle.load(f)

        preprocessor = joblib.load('full_preprocessor.joblib')
        model = load_model('lstm_model.keras')
        scaler_y = joblib.load('scaler_y.joblib')

        return model, preprocessor, scaler_y, features
    except Exception as e:
        print(f"Ошибка загрузки: {e}")
        raise

In [ ]:
def predict_cost(input_dict, model, preprocessor, scaler_y, features):
    """
    input_dict: словарь с данными вида {
        'Год': 2025,
        'Месяц': 6,
        'День': 1,
        'Проходной балл': 85,
        'Бюджетные места': 20,
        'Программа': 'Информатика',
        'Вуз': 'МГУ'
    }
    """
    try:
        # Создаем DataFrame с правильным порядком колонок
        input_df = pd.DataFrame([input_dict])[features]

        # Преобразуем данные
        X_processed = preprocessor.transform(input_df)
        if hasattr(X_processed, 'toarray'):
            X_processed = X_processed.toarray()

        # Проверяем размерность
        if X_processed.shape[1] != model.input_shape[2]:
            raise ValueError(
                f"Несоответствие размеров: модель ожидает {model.input_shape[2]} признаков, "
                f"получено {X_processed.shape[1]}. Проверьте препроцессор."
            )

        X_reshaped = np.expand_dims(X_processed, axis=1)
        y_pred = scaler_y.inverse_transform(model.predict(X_reshaped))
        return y_pred[0][0]
    except Exception as e:
        print(f"Ошибка предсказания: {e}")
        return None

In [ ]:
# Загрузка компонентов
model, preprocessor, scaler_y, features = load_model_components()

# Входные данные
input_data = {
    'Год': 2024,
    'Месяц': 11,
    'День': 1,
    'Проходной балл': 30,
    'Бюджетные места': 20,
    'Программа': 'Психология',
    'Вуз': 'нет вуза'
}

# Предсказание
prediction = predict_cost(input_data, model, preprocessor, scaler_y, features)
if prediction is not None:
    print(f"Предсказанная стоимость: {prediction:.2f} руб.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
Предсказанная стоимость: 243085.05 руб.
